## Cleaning text + basic structuring
### Step 1 : Load the Dataset

In [3]:
# Load sample csv
import pandas as pd
df = pd.read_csv("../data/twcs_small.csv")
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0
1,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN
2,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0
3,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0
4,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN


In [6]:
# check text column
df.text

0       @161252 What's that egg website people talk about
1       Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...
2       @693975 We can assist you. We recommend updati...
3       @331912 @115955 Thats better than having an un...
4       @VirginAmerica is probably one of the best air...
                              ...                        
4995    @407091 Hello, Were you able to get your inqui...
4996    @139084 Great. Let us know via DM how things i...
4997    .@XboxSupport: the new #XboxOne update appears...
4998    @170844 @AmazonHelp @AmazonHelp , can u plz ch...
4999    @116618 Is it possible to have a custom main p...
Name: text, Length: 5000, dtype: str

### Step 2 — Clean the text
Remove:
- URLs
- Mentions (@brand)
- Hashtags
- Emojis
- Punctuation
- Extra spaces

## Create a classic text normalization function with the following action:
- Convert to Lowercase  
- Remove URLs : re.sub(r"http\S+"," " ,text)   
      - http → literal characters “http”  
      - \S+ → one or more non‑space characters
#### Meaning: Match anything that starts with http along with non white space characters and continues until the next whitespace.

- Remove mentions: re.sub(r"@\w+", " ", text)   
      - @ → literal @  
      - \w+ → one or more `“word characters”`: letters (a–z), digits (0–9), underscore (_)
#### Meaning: Match anything that starts with an @ along with word characters and continues until the next whitespace.

- Remove hashtags :re.sub(r"#\w+", " ", text)  
      - # → literal #  
      - \w+ → letters, digits, underscore  
#### Meaning: Match anything that starts with a # along with word characters and continues until the next whitespace.

- Remove punctuation/emojis: re.sub(r"[^a-z0-9\s]", " ", text)  
      - [...] → character class  
      - ^ → negation (means “anything not inside this list”)  
      - a-z → lowercase letters  
      - 0-9 → digits  
      - \s → whitespace (space, tab, newline)  
#### Meaning: Replace anything that is NOT: a lowercase letter, a digit , whitespace, with a space.
#### Examples removed:  
- punctuation: ! ? , . ; :  
- emojis: 😀🔥💯  
- symbols: $ % ^ & *  
- non‑English characters: é, ü, 你好  

- Collapse whitespace: re.sub(r"\s+", " ", text).strip()  
      - \s → whitespace  
      - + → one or more  
#### Meaning: Replace multiple spaces/tabs/newlines with a single space.  
- .strip() removes leading/trailing spaces.

  
This is a common preprocessing pipeline for NLP tasks like sentiment analysis, topic modeling, or training ML models.

In [10]:
# create a regex function
import re

def clean_text(text):
    # convert to lowercase
    text = text.lower()
    # apply regex to remove http with one space untill next whitespace
    text = re.sub(r"http\S+"," " ,text)
    # apply regex to remove @ mention + characters with one space untill next whitespace
    text = re.sub(r"@\w+"," " ,text)
    # apply regex to remove # mentions + characters with one space untill next whitespace
    text = re.sub(r"#\w+", " ", text)
    # apply regex to remove any characters not in regex list
    text = re.sub(r"[^a-z0-9\s]", " " , text)
    # apply regex to remove additional whitespaces , tabs and newline with a space and strip trailing white spaces
    text = re.sub(r"\s+", " " , text).strip()
    return text 
    
    

In [12]:
# call the function
df['Clean_text'] = df['text'].apply(clean_text)
# show the columns side by side
df[['Clean_text', 'text']].head()

,Clean_text,text
0,what s that egg website people talk about,@161252 What's that egg website people talk about
1,why,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...
2,we can assist you we recommend updating to ios...,@693975 We can assist you. We recommend updati...
3,thats better than having an unstable connectio...,@331912 @115955 Thats better than having an un...
4,is probably one of the best airlines i ve ever...,@VirginAmerica is probably one of the best air...


## Step 3 - Find and fill conversation ID

If a tweet is replying to another → conversation_id = parent tweet  
If not → conversation_id = its own tweet_id


In [13]:
df.in_response_to_tweet_id.isna()

0       False
1        True
2       False
3       False
4        True
        ...  
4995    False
4996    False
4997     True
4998    False
4999     True
Name: in_response_to_tweet_id, Length: 5000, dtype: bool

In [14]:
df.in_response_to_tweet_id.isna().sum()

np.int64(1395)

In [18]:
# Fill with twitter id
df["conversation_id"] = df["in_response_to_tweet_id"].fillna(df["tweet_id"])
df[["conversation_id","in_response_to_tweet_id"]].head()

,conversation_id,in_response_to_tweet_id
0,192625.0,192625.0
1,738238.0,NaN
2,2414304.0,2414304.0
3,1793930.0,1793930.0
4,2088018.0,NaN


In [16]:
df["conversation_id"].isna()

0       False
1       False
2       False
3       False
4       False
        ...  
4995    False
4996    False
4997    False
4998    False
4999    False
Name: conversation_id, Length: 5000, dtype: bool

## Step 5 — Save cleaned dataset

In [17]:
df.to_csv("../data/twcs_clean.csv", index=False)